# Stenosis Detection — Kaggle (plug & play)

**YOLO11s @ 768px on ARCADE task-2 + Danilov (capped) + CADICA.** Target **F1 > 0.57** (recall-weighted).

## Do this first, then Run All
1. **Settings → Accelerator → GPU** (T4).
2. **Settings → Internet → ON** (pip + git clone).
3. **+ Add Input**: attach **ARCADE**, **danilov-stenosis** (title contains *danilov*), and **CADICA** (title contains *cadica*). CADICA adds patient diversity — the #1 accuracy lever. Any subset works; missing ones auto-skip.
4. **Run All.** First pass keeps `DRY_RUN = True` (3-epoch wiring check). Then set `DRY_RUN = False` and Run All again for the real 150-epoch run.

Everything else auto-detects (dataset paths at any depth, OOM fallback to batch 8, output collection). No paths to edit.

**Honesty + safety baked in:** a **leakage audit** (§3b) runs *before* training and **hard-fails** if any patient/clip lands in both train and val. Danilov is **capped to 5 frames/patient** (its 8325 frames are only 64 patients — redundant). Training enforces a **recall-first F1 gate** (a missed stenosis is the deadly error) and eval runs at low conf so recall isn't throttled. After training, §5 writes a **val-only GT-vs-pred demo** to `outputs/stenosis_demo.mp4`.

## 1 · GPU + clone + install

In [ ]:
import os, sys, torch
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU.'
print('CUDA:', torch.cuda.get_device_name(0))

REPO_SLUG    = 'jugalmodi0111/interventional-imaging-pipeline'
GITHUB_TOKEN = ''                              # '' if repo is PUBLIC, else paste a PAT
REPO         = '/kaggle/tmp/interventional-imaging-pipeline'  # /kaggle/tmp NOT saved to output -> repo+9k PNGs stay out of download
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone -q https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install 'ultralytics>=8.2' pycocotools opencv-python pyyaml
from src import env
E = env.setup()
torch.backends.cudnn.benchmark = True

## 2 · Auto-detect + link the attached datasets
No paths to edit — finds ARCADE and any *danilov* folder at any depth under `/kaggle/input`.

In [ ]:
import glob, os
# ARCADE: the folder that CONTAINS stenosis/*/annotations/*.json
ah = glob.glob('/kaggle/input/**/stenosis/*/annotations/*.json', recursive=True)
assert ah, 'No ARCADE stenosis json under /kaggle/input — + Add Input the ARCADE dataset.'
ARCADE_INPUT = ah[0].split('/stenosis/')[0]

def _find(word):                                          # any *word* dir at ANY depth (shallowest match)
    hits = [os.path.normpath(d) for d in glob.glob('/kaggle/input/**/', recursive=True)
            if word in os.path.basename(os.path.normpath(d)).lower()]
    return min(hits, key=len) if hits else ''

DANILOV_INPUT = _find('danilov')
CADICA_INPUT  = _find('cadica')                           # patient-diverse add (folder holds selectedVideos/)

print('inputs attached:', os.listdir('/kaggle/input'))
print('ARCADE_INPUT  =', ARCADE_INPUT)
print('DANILOV_INPUT =', DANILOV_INPUT or '(none)')
print('CADICA_INPUT  =', CADICA_INPUT or '(none)')

os.makedirs('data/raw', exist_ok=True)
os.system(f'ln -sfn "{ARCADE_INPUT}" data/raw/arcade')
if DANILOV_INPUT:
    os.system(f'ln -sfn "{DANILOV_INPUT}" data/raw/danilov')
if CADICA_INPUT:
    os.system(f'ln -sfn "{CADICA_INPUT}" data/raw/cadica')
assert glob.glob('data/raw/arcade/stenosis/**/*.json', recursive=True), 'ARCADE symlink wrong'
if DANILOV_INPUT:
    print('danilov bmp:', len(glob.glob('data/raw/danilov/**/*.bmp', recursive=True)),
          '| xml:', len(glob.glob('data/raw/danilov/**/*.xml', recursive=True)))
if CADICA_INPUT:
    print('cadica groundtruth files:',
          len(glob.glob('data/raw/cadica/**/groundtruth/*.txt', recursive=True)))

## 3 · Standardize ARCADE (+ Danilov) → YOLO

In [ ]:
import yaml, glob
cfg = yaml.safe_load(open('configs/stenosis_yolo.yaml'))
if not DANILOV_INPUT:
    cfg['datasets'].pop('danilov', None)                  # ARCADE-only if Danilov not attached
if not CADICA_INPUT:
    cfg['datasets'].pop('cadica', None)                   # skip CADICA if not attached
from src.data_prep import danilov_to_yolo, cadica_to_yolo
danilov_to_yolo.main(cfg)                                  # ARCADE (+ Danilov, capped 5/patient via max_frames_per_patient)
if CADICA_INPUT:
    print('cadica:', cadica_to_yolo.main(cfg), 'frames')   # patient-diverse add (keyframes only, patient-grouped split)
ntr = len(glob.glob('data/processed/stenosis/images/train/*'))
nva = len(glob.glob('data/processed/stenosis/images/val/*'))
print(f'train imgs: {ntr} | val imgs: {nva}')
assert ntr and nva, 'conversion produced no images — check the printed per-dataset counts above'

## 3b · Leakage audit (hard gate)
Verifies the split is **patient-grouped**, not per-frame — the check that makes the F1 trustworthy. Runs **before** training and **raises** (stops Run All) if either:
1. any patient/clip group appears in **both** train and val, or
2. the Danilov frames were **not** collapsed by `group_key` (their filenames don't match the expected `<site>_<patient>_<seq>_<frame>` pattern → silent per-frame leak).

If this cell raises, **the reported F1 would be inflated** — fix the split before trusting any number.

In [ ]:
# HARD GATE — no leak, no false F1. Build the TRUE Danilov stem set from raw (independent of the
# regex) so a silent grouping no-op is caught, then audit the produced YOLO split.
import os
from src.data_prep.io_utils import audit_split_leakage

_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
danilov_stems = None
if DANILOV_INPUT:
    danilov_stems = {os.path.splitext(f)[0]
                     for _, _, fs in os.walk('data/raw/danilov') for f in fs
                     if os.path.splitext(f)[1].lower() in _EXTS}

rep = audit_split_leakage('data/processed/stenosis', danilov_stems=danilov_stems)  # raises on any leak
print('LEAKAGE CHECK PASSED — split is patient-grouped and honest')
print(f"  train {rep['train_imgs']} imgs / {rep['train_groups']} groups | "
      f"val {rep['val_imgs']} imgs / {rep['val_groups']} groups "
      f"(val ~{rep['val_frac_by_group']:.0%} by group)")
if rep['danilov']:
    d = rep['danilov']
    print(f"  danilov: {d['danilov_frames']} frames -> {d['patient_groups']} patients, "
          f"{d['ungrouped']} ungrouped ({d['ungrouped_frac']:.0%})")

# Extra integrity checks (WARN — surface silent-but-bad training conditions):
from src.data_prep.io_utils import duplicate_basenames_across_cocos
dupes = duplicate_basenames_across_cocos('data/raw/arcade/stenosis')
if dupes:
    print(f'\n[info] ARCADE cross-split basename repeats: {len(dupes)} names in >1 split '
          f'-> auto-disambiguated by split prefix in coco_to_yolo (no data loss).')
    print('       e.g.', list(dupes)[:5])
vg = rep['val_groups']
if vg < 10:
    print(f'\n[WARN] only {vg} distinct patient/clip groups in val -> F1 will be noisy; '
          'treat the number as a wide estimate.')

## 4 · Train YOLO11s @ 768
Leave `DRY_RUN = True` for a 3-epoch wiring check. Set `False` for the real 150-epoch run (target **F1 > 0.57**). OOM auto-falls back to batch 8.

`train()` now **computes and prints F1** (recall-weighted from the PR curve) and an explicit `[PASS]/[FAIL] F1 vs floor 0.57` line — the F1 floor is now enforced/surfaced, not just mAP50. SSL (pseudo-label + gdino) is also guarded inside `train()`: it is skipped unless a disjoint `ssl.unlabeled_dir` exists, so it can't re-leak val frames.

In [ ]:
import os, glob
DRY_RUN = False         # honest re-run: patient-grouped split (io_utils.group_key). Run All.

if DRY_RUN:
    cfg['train']['epochs'] = 3
    cfg['ssl'] = {'pseudo_label': False}                  # skip SSL on the fast pass

# SSL pseudo-labeling copies ALL ssl.unlabeled_dir frames into images/train. If that dir holds val
# patients' frames it re-leaks them past the audit. Only allow SSL when a disjoint, existing
# unlabeled dir is actually attached; otherwise disable it (default here -> no leak).
_ssl = cfg.get('ssl', {})
_udir = _ssl.get('unlabeled_dir')
_has_unlabeled = bool(_udir and os.path.isdir(_udir)
                      and glob.glob(os.path.join(_udir, '**', '*.png'), recursive=True))
# Both pseudo-label AND the gdino cold-start copy every unlabeled_dir frame into images/train.
# Without a disjoint unlabeled set that silently re-leaks val patients -> disable both.
if not _has_unlabeled and (_ssl.get('pseudo_label') or _ssl.get('seed') == 'gdino'):
    cfg['ssl'] = dict(_ssl, pseudo_label=False, seed=None)
    print('SSL disabled (pseudo-label + gdino seed): no disjoint ssl.unlabeled_dir attached '
          '-> would re-leak val frames into train. Attach an unlabeled-only dir (e.g. XCAD) to enable.')

from src.train.train_detector import train, train_kwargs, run_tag
TAG = run_tag(cfg)
print('run:', TAG, '| kwargs ->', train_kwargs(cfg))
try:
    best = train(cfg, project=f"{E['runs']}/stenosis/{TAG}")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        torch.cuda.empty_cache(); cfg['train']['batch'] = 8
        print('CUDA OOM at batch 16 -> retrying at batch 8'); best = train(cfg, project=f"{E['runs']}/stenosis/{TAG}")
    else:
        raise
print('best weights ->', best)

## 5 · Demo video — GT vs prediction on held-out (val) patients
Runs `best.pt` over the **val** split (unseen patients) and writes an overlay mp4:
**green = model prediction (+conf), red = ground truth.** Saved to `outputs/stenosis_demo.mp4` and copied to `/kaggle/working` for download. Val-only → an honest look at generalization, not the (now impossible) leaked frames.

In [ ]:
import os, glob, shutil, cv2
from ultralytics import YOLO

VAL_IMG, VAL_LBL = 'data/processed/stenosis/images/val', 'data/processed/stenosis/labels/val'
frames = sorted(glob.glob(os.path.join(VAL_IMG, '*.png')))[:400]   # cap demo length
assert frames, 'no val frames to render'
os.makedirs('outputs', exist_ok=True)
DEMO = 'outputs/stenosis_demo.mp4'
model = YOLO(best)
h, w = cv2.imread(frames[0]).shape[:2]
vw = cv2.VideoWriter(DEMO, cv2.VideoWriter_fourcc(*'mp4v'), 12, (w, h))
for fp in frames:
    img = cv2.imread(fp)                                          # already CLAHE'd 768 png
    H, W = img.shape[:2]
    lp = os.path.join(VAL_LBL, os.path.splitext(os.path.basename(fp))[0] + '.txt')
    if os.path.exists(lp):                                        # ground truth -> red
        for ln in open(lp):
            p = ln.split()
            if len(p) == 5:
                cx, cy, bw, bh = (float(v) for v in p[1:])
                cv2.rectangle(img, (int((cx-bw/2)*W), int((cy-bh/2)*H)),
                              (int((cx+bw/2)*W), int((cy+bh/2)*H)), (0, 0, 200), 2)
    r = model.predict(fp, conf=0.25, imgsz=W, verbose=False)[0]   # prediction -> green
    for b in r.boxes:
        x1, y1, x2, y2 = (int(v) for v in b.xyxy[0].tolist())
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(img, f'{float(b.conf[0]):.2f}', (x1, max(12, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
    cv2.putText(img, 'green=pred  red=GT  (val / unseen patients)', (8, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    vw.write(img)
vw.release()
if os.path.isdir('/kaggle/working'):
    shutil.copy(DEMO, '/kaggle/working/stenosis_demo.mp4')
print('demo video ->', DEMO, '| frames:', len(frames))

## 6 · Collect outputs for download
Copies `best.pt` + `results.csv` to `/kaggle/working` and zips the full run. Grab them from the **Output** panel (right).

In [ ]:
import shutil, os
run_dir = os.path.dirname(os.path.dirname(best))          # .../<TAG>/base
for rel in ('weights/best.pt', 'results.csv'):
    src = os.path.join(run_dir, rel)
    if os.path.exists(src):
        dst = f'/kaggle/working/{os.path.basename(rel)}'
        shutil.copy(src, dst); print('download ->', dst)
shutil.make_archive('/kaggle/working/stenosis_run', 'zip', run_dir)
print('full run zip -> /kaggle/working/stenosis_run.zip')
print('\nNext (on your Mac): python -m src.export.yolo_to_coreml --weights best.pt')